# 🧲 Computational Discovery of High-Temperature Soft Magnets
## MEN 6810 — Full ML Pipeline with AI-Driven Analysis

**Pipeline Overview:**
1. Data Loading & Cleaning
2. Exploratory Data Analysis (EDA)
3. Baseline Models (Sparse Data)
4. Rigorous Filtering + Feature Engineering
5. Optimized Models (RF + XGBoost + Bayesian Hypertuning)
6. SHAP Explainability
7. Uncertainty Quantification
8. Virtual Synthesis & Candidate Discovery
9. Publication-Quality Figures


In [ ]:
# ============================================================
# STEP 1: Install Dependencies (run once)
# ============================================================
!pip install xgboost shap scikit-learn matplotlib seaborn pandas numpy scipy -q
print("✅ All packages installed.")


In [ ]:
# ============================================================
# STEP 2: Load Data + Apply Cleaning Fixes
# ============================================================
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ------ UPLOAD magneticMaterials_finalDataset_encoded.csv to Colab first ------
# Then update this path:
df = pd.read_csv('/content/magneticMaterials_finalDataset_encoded.csv')
print(f"Raw loaded shape: {df.shape}")

# --- Fix 1: Remove duplicates ---
n_before = len(df)
df = df.drop_duplicates()
print(f"Removed {n_before - len(df)} duplicate rows → {len(df)} remaining")

# --- Fix 2: Remove row with density = 0 (physically impossible) ---
n_before = len(df)
df = df[df['rho_theoretical (g/cm^3)'] > 0]
print(f"Removed {n_before - len(df)} rows with zero density → {len(df)} remaining")

# --- Fix 3: Convert boolean columns to int (0/1) for model compatibility ---
bool_cols = df.select_dtypes(include='bool').columns.tolist()
df[bool_cols] = df[bool_cols].astype(int)
print(f"Converted {len(bool_cols)} boolean columns to int")

# --- Fix 4: Merge rare Magnetic_type categories (<5 samples) into 'Other' ---
magnetic_type_cols = [c for c in df.columns if c.startswith('Magnetic_type_')]
rare_cols = []
for c in magnetic_type_cols:
    if df[c].sum() < 5:
        rare_cols.append(c)

# Create 'Other' column = 1 if any rare category is 1
df['Magnetic_type_Other'] = df[rare_cols].max(axis=1)
df = df.drop(columns=rare_cols)
print(f"Merged {len(rare_cols)} rare Magnetic_type categories into 'Other'")

# --- Fix 5: Drop element columns that are never used (all zeros) ---
x_cols = [c for c in df.columns if c.startswith('x_')]
zero_cols = [c for c in x_cols if df[c].sum() == 0]
df = df.drop(columns=zero_cols)
print(f"Dropped {len(zero_cols)} all-zero element columns")

print(f"\n✅ Final clean dataset: {df.shape}")
print(f"   Targets: Coercivity (Oe), Curie(TC) (K)")
print(f"   Features: {df.shape[1] - 2}")
df.head()


In [ ]:
# ============================================================
# STEP 3: Exploratory Data Analysis (EDA)
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# 1. Curie Temperature Distribution
axes[0,0].hist(df['Curie(TC) (K)'], bins=50, color='#2196F3', edgecolor='black', alpha=0.8)
axes[0,0].axvline(x=1000, color='red', linestyle='--', linewidth=2, label='Target: 1000 K')
axes[0,0].set_xlabel('Curie Temperature (K)', fontsize=12)
axes[0,0].set_ylabel('Count', fontsize=12)
axes[0,0].set_title('Distribution of Curie Temperature', fontsize=13, fontweight='bold')
axes[0,0].legend(fontsize=10)

# 2. Coercivity Distribution (log scale)
log_hc = np.log10(df['Coercivity (Oe)'].clip(lower=0.01))
axes[0,1].hist(log_hc, bins=50, color='#FF9800', edgecolor='black', alpha=0.8)
axes[0,1].axvline(x=2, color='red', linestyle='--', linewidth=2, label='Target: 100 Oe')
axes[0,1].set_xlabel('log₁₀(Coercivity) (Oe)', fontsize=12)
axes[0,1].set_ylabel('Count', fontsize=12)
axes[0,1].set_title('Distribution of Coercivity (Log Scale)', fontsize=13, fontweight='bold')
axes[0,1].legend(fontsize=10)

# 3. VEC vs Curie Temperature
scatter = axes[0,2].scatter(df['VEC'], df['Curie(TC) (K)'], c=log_hc, 
                            cmap='coolwarm', s=15, alpha=0.6, edgecolors='none')
axes[0,2].set_xlabel('Valence Electron Concentration (VEC)', fontsize=12)
axes[0,2].set_ylabel('Curie Temperature (K)', fontsize=12)
axes[0,2].set_title('VEC vs Curie Temp (color = log Hc)', fontsize=13, fontweight='bold')
plt.colorbar(scatter, ax=axes[0,2], label='log₁₀(Hc)')

# 4. Density vs Curie Temperature  
axes[1,0].scatter(df['rho_theoretical (g/cm^3)'], df['Curie(TC) (K)'], 
                  s=15, alpha=0.5, color='#4CAF50')
axes[1,0].set_xlabel('Theoretical Density (g/cm³)', fontsize=12)
axes[1,0].set_ylabel('Curie Temperature (K)', fontsize=12)
axes[1,0].set_title('Density vs Curie Temperature', fontsize=13, fontweight='bold')

# 5. Soft vs Hard magnet distribution
magnet_counts = pd.Series({
    'Hard': df['Magnet_type_Hard'].sum(),
    'Soft': df['Magnet_type_Soft'].sum(),
    'Semi-Hard': df.get('Magnet_type_Semi-Hard', pd.Series([0])).sum(),
    'Hard/Soft': df.get('Magnet_type_Hard/Soft', pd.Series([0])).sum()
})
magnet_counts = magnet_counts[magnet_counts > 0]
axes[1,1].bar(magnet_counts.index, magnet_counts.values, color=['#E53935','#43A047','#FB8C00','#8E24AA'], edgecolor='black')
axes[1,1].set_ylabel('Count', fontsize=12)
axes[1,1].set_title('Magnet Type Distribution', fontsize=13, fontweight='bold')

# 6. Correlation of top features with Tc
x_cols = [c for c in df.columns if c.startswith('x_')]
corr_tc = df[x_cols + ['VEC', 'rho_theoretical (g/cm^3)']].corrwith(df['Curie(TC) (K)']).abs().sort_values(ascending=False).head(15)
axes[1,2].barh(corr_tc.index[::-1], corr_tc.values[::-1], color='#7B1FA2')
axes[1,2].set_xlabel('|Pearson Correlation| with Tc', fontsize=12)
axes[1,2].set_title('Top Feature Correlations with Curie Temp', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/fig1_EDA.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Figure 1 saved: fig1_EDA.png")


In [ ]:
# ============================================================
# STEP 4: Define Features & Targets
# ============================================================
from sklearn.model_selection import train_test_split

# Targets
yTc = df['Curie(TC) (K)']
yHc_log = np.log10(df['Coercivity (Oe)'].clip(lower=0.01))  # log-transform for stability

# Features = everything except targets
X = df.drop(columns=['Coercivity (Oe)', 'Curie(TC) (K)'])

# Train/Test split (80/20, stratified by binned Tc for balanced distribution)
tc_bins = pd.qcut(yTc, q=5, labels=False, duplicates='drop')

X_train, X_test, yTc_train, yTc_test, yHc_train, yHc_test = train_test_split(
    X, yTc, yHc_log, test_size=0.2, random_state=42, stratify=tc_bins
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set:     {X_test.shape[0]} samples")
print(f"Features:     {X_train.shape[1]}")
print(f"\nTargets:")
print(f"  Curie Temp (K) - train range: [{yTc_train.min():.1f}, {yTc_train.max():.1f}]")
print(f"  log₁₀(Hc) - train range: [{yHc_train.min():.2f}, {yHc_train.max():.2f}]")


In [ ]:
# ============================================================
# STEP 5: Train Optimized XGBoost for Curie Temperature
# ============================================================
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

# Wider hyperparameter search space than basic GridSearch
xgb_param_dist = {
    'n_estimators': randint(200, 1000),
    'max_depth': randint(3, 12),
    'learning_rate': uniform(0.01, 0.29),   # 0.01 to 0.30
    'subsample': uniform(0.6, 0.4),          # 0.6 to 1.0
    'colsample_bytree': uniform(0.5, 0.5),   # 0.5 to 1.0
    'min_child_weight': randint(1, 10),
    'reg_alpha': uniform(0, 1),
    'reg_lambda': uniform(0.5, 2),
    'gamma': uniform(0, 0.5)
}

xgb_tc = XGBRegressor(random_state=42, n_jobs=-1, tree_method='hist')

xgb_search_tc = RandomizedSearchCV(
    xgb_tc, xgb_param_dist, 
    n_iter=100,       # 100 random combinations
    cv=5, 
    scoring='r2', 
    random_state=42, 
    n_jobs=-1, 
    verbose=1
)

xgb_search_tc.fit(X_train, yTc_train)

print(f"\n✅ Best XGBoost Curie Temp R² (CV): {xgb_search_tc.best_score_:.4f}")
print(f"Best params: {xgb_search_tc.best_params_}")
best_xgb_tc = xgb_search_tc.best_estimator_


In [ ]:
# ============================================================
# STEP 6: Train Optimized Random Forest for Coercivity (log)
# ============================================================
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

rf_param_dist = {
    'n_estimators': randint(200, 800),
    'max_depth': [10, 20, 30, 50, None],
    'min_samples_split': randint(2, 10),
    'min_samples_leaf': randint(1, 5),
    'max_features': ['sqrt', 'log2', 0.3, 0.5, 0.7],
    'bootstrap': [True, False]
}

rf_hc = RandomForestRegressor(random_state=42, n_jobs=-1)

rf_search_hc = RandomizedSearchCV(
    rf_hc, rf_param_dist,
    n_iter=80,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=1
)

rf_search_hc.fit(X_train, yHc_train)

print(f"\n✅ Best RF Coercivity R² (CV): {rf_search_hc.best_score_:.4f}")
print(f"Best params: {rf_search_hc.best_params_}")
best_rf_hc = rf_search_hc.best_estimator_


In [ ]:
# ============================================================
# STEP 7: Train All 4 Model Combinations for Comparison Table
# ============================================================
# XGBoost for Coercivity
xgb_hc = XGBRegressor(**xgb_search_tc.best_params_, random_state=42, n_jobs=-1)
xgb_hc.fit(X_train, yHc_train)

# RF for Curie Temp
rf_tc = RandomForestRegressor(**rf_search_hc.best_params_, random_state=42, n_jobs=-1)
rf_tc.fit(X_train, yTc_train)

print("✅ All 4 models trained:")
print("  1. XGBoost → Curie Temp (primary)")
print("  2. RF → log(Coercivity) (primary)")
print("  3. XGBoost → log(Coercivity) (comparison)")
print("  4. RF → Curie Temp (comparison)")


In [ ]:
# ============================================================
# STEP 8: Evaluate All Models — R², MAE, RMSE
# ============================================================
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

def evaluate(name, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {'Model': name, 'R²': r2, 'MAE': mae, 'RMSE': rmse}

# Predictions
pred_xgb_tc = best_xgb_tc.predict(X_test)
pred_rf_tc  = rf_tc.predict(X_test)
pred_rf_hc  = best_rf_hc.predict(X_test)
pred_xgb_hc = xgb_hc.predict(X_test)

results = pd.DataFrame([
    evaluate('XGBoost → Curie Temp (K)', yTc_test, pred_xgb_tc),
    evaluate('RF → Curie Temp (K)', yTc_test, pred_rf_tc),
    evaluate('RF → log₁₀(Hc)', yHc_test, pred_rf_hc),
    evaluate('XGBoost → log₁₀(Hc)', yHc_test, pred_xgb_hc),
])

print("=" * 65)
print("MODEL COMPARISON — TEST SET PERFORMANCE")
print("=" * 65)
print(results.to_string(index=False, float_format='{:.4f}'.format))
print("=" * 65)

# ---- Actual vs Predicted Plots (Publication Quality) ----
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Curie Temp
axes[0].scatter(yTc_test, pred_xgb_tc, s=20, alpha=0.6, c='#1565C0', edgecolors='none', label=f'XGBoost (R²={r2_score(yTc_test, pred_xgb_tc):.3f})')
axes[0].scatter(yTc_test, pred_rf_tc, s=20, alpha=0.4, c='#E65100', edgecolors='none', marker='s', label=f'RF (R²={r2_score(yTc_test, pred_rf_tc):.3f})')
lims = [0, 1500]
axes[0].plot(lims, lims, 'k--', lw=1.5, alpha=0.5)
axes[0].set_xlim(lims); axes[0].set_ylim(lims)
axes[0].set_xlabel('Actual Curie Temperature (K)', fontsize=13)
axes[0].set_ylabel('Predicted Curie Temperature (K)', fontsize=13)
axes[0].set_title('Curie Temperature Prediction', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=11)

# Coercivity (log)
axes[1].scatter(yHc_test, pred_rf_hc, s=20, alpha=0.6, c='#2E7D32', edgecolors='none', label=f'RF (R²={r2_score(yHc_test, pred_rf_hc):.3f})')
axes[1].scatter(yHc_test, pred_xgb_hc, s=20, alpha=0.4, c='#AD1457', edgecolors='none', marker='s', label=f'XGBoost (R²={r2_score(yHc_test, pred_xgb_hc):.3f})')
lims2 = [-2.5, 5.5]
axes[1].plot(lims2, lims2, 'k--', lw=1.5, alpha=0.5)
axes[1].set_xlim(lims2); axes[1].set_ylim(lims2)
axes[1].set_xlabel('Actual log₁₀(Coercivity) (Oe)', fontsize=13)
axes[1].set_ylabel('Predicted log₁₀(Coercivity) (Oe)', fontsize=13)
axes[1].set_title('Coercivity Prediction (Log Scale)', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=11)

plt.tight_layout()
plt.savefig('/content/fig2_actual_vs_predicted.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Figure 2 saved: fig2_actual_vs_predicted.png")


In [ ]:
# ============================================================
# STEP 9: Residual Analysis
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tc residuals
resid_tc = yTc_test.values - pred_xgb_tc
axes[0].scatter(pred_xgb_tc, resid_tc, s=15, alpha=0.5, c='#1565C0', edgecolors='none')
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Predicted Curie Temperature (K)', fontsize=12)
axes[0].set_ylabel('Residual (K)', fontsize=12)
axes[0].set_title('Residuals — XGBoost Curie Temp', fontsize=13, fontweight='bold')

# Hc residuals
resid_hc = yHc_test.values - pred_rf_hc
axes[1].scatter(pred_rf_hc, resid_hc, s=15, alpha=0.5, c='#2E7D32', edgecolors='none')
axes[1].axhline(y=0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Predicted log₁₀(Coercivity)', fontsize=12)
axes[1].set_ylabel('Residual', fontsize=12)
axes[1].set_title('Residuals — RF log(Coercivity)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/fig3_residuals.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Figure 3 saved: fig3_residuals.png")


In [ ]:
# ============================================================
# STEP 10: Learning Curves (Train vs Validation)
# ============================================================
from sklearn.model_selection import learning_curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, model, y_train_data, title, color in [
    (axes[0], best_xgb_tc, yTc_train, 'XGBoost — Curie Temp', '#1565C0'),
    (axes[1], best_rf_hc, yHc_train, 'RF — log(Coercivity)', '#2E7D32')
]:
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_train, y_train_data,
        train_sizes=np.linspace(0.1, 1.0, 10),
        cv=5, scoring='r2', n_jobs=-1, random_state=42
    )
    
    train_mean = train_scores.mean(axis=1)
    train_std  = train_scores.std(axis=1)
    val_mean   = val_scores.mean(axis=1)
    val_std    = val_scores.std(axis=1)
    
    ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color=color)
    ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='#E65100')
    ax.plot(train_sizes, train_mean, 'o-', color=color, label='Train R²', markersize=4)
    ax.plot(train_sizes, val_mean, 's-', color='#E65100', label='Validation R²', markersize=4)
    ax.set_xlabel('Training Set Size', fontsize=12)
    ax.set_ylabel('R² Score', fontsize=12)
    ax.set_title(f'Learning Curve — {title}', fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.set_ylim([0, 1.05])
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/fig4_learning_curves.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Figure 4 saved: fig4_learning_curves.png")


In [ ]:
# ============================================================
# STEP 11: SHAP Analysis — Curie Temperature (THE AI STEP)
# ============================================================
import shap

# Use TreeExplainer for XGBoost (fast, exact)
explainer_tc = shap.TreeExplainer(best_xgb_tc)
shap_values_tc = explainer_tc.shap_values(X_test)

# --- SHAP Summary (Beeswarm) Plot ---
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_tc, X_test, max_display=20, show=False, plot_size=(10, 8))
plt.title('SHAP Feature Importance — Curie Temperature', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('/content/fig5_shap_tc_summary.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Figure 5 saved: fig5_shap_tc_summary.png")


In [ ]:
# ============================================================
# STEP 12: SHAP Analysis — Coercivity
# ============================================================
explainer_hc = shap.TreeExplainer(best_rf_hc)
shap_values_hc = explainer_hc.shap_values(X_test)

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_hc, X_test, max_display=20, show=False, plot_size=(10, 8))
plt.title('SHAP Feature Importance — log(Coercivity)', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('/content/fig6_shap_hc_summary.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Figure 6 saved: fig6_shap_hc_summary.png")


In [ ]:
# ============================================================
# STEP 13: SHAP Dependence Plots — Key Physical Insights
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Curie Temp dependence
shap.dependence_plot('x_Fe', shap_values_tc, X_test, ax=axes[0,0], show=False)
axes[0,0].set_title('Fe content → Curie Temp', fontsize=12, fontweight='bold')

shap.dependence_plot('VEC', shap_values_tc, X_test, ax=axes[0,1], show=False)
axes[0,1].set_title('VEC → Curie Temp', fontsize=12, fontweight='bold')

shap.dependence_plot('x_Co', shap_values_tc, X_test, ax=axes[1,0], show=False)
axes[1,0].set_title('Co content → Curie Temp', fontsize=12, fontweight='bold')

shap.dependence_plot('rho_theoretical (g/cm^3)', shap_values_tc, X_test, ax=axes[1,1], show=False)
axes[1,1].set_title('Density → Curie Temp', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/fig7_shap_dependence.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Figure 7 saved: fig7_shap_dependence.png")


In [ ]:
# ============================================================
# STEP 14: Uncertainty Quantification via Random Forest Variance
# ============================================================
# RF naturally provides UQ: each tree gives a prediction
# Standard deviation across trees = prediction uncertainty

# For Curie Temp (use the RF model)
tree_preds_tc = np.array([tree.predict(X_test.values) for tree in rf_tc.estimators_])
pred_tc_mean = tree_preds_tc.mean(axis=0)
pred_tc_std  = tree_preds_tc.std(axis=0)

# For Coercivity
tree_preds_hc = np.array([tree.predict(X_test.values) for tree in best_rf_hc.estimators_])
pred_hc_mean = tree_preds_hc.mean(axis=0)
pred_hc_std  = tree_preds_hc.std(axis=0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tc with error bars
sort_idx = np.argsort(yTc_test.values)
x_plot = np.arange(len(sort_idx))
axes[0].fill_between(x_plot, 
                     (pred_tc_mean[sort_idx] - 2*pred_tc_std[sort_idx]),
                     (pred_tc_mean[sort_idx] + 2*pred_tc_std[sort_idx]),
                     alpha=0.3, color='#1565C0', label='95% CI (±2σ)')
axes[0].scatter(x_plot, yTc_test.values[sort_idx], s=8, c='black', zorder=3, label='Actual', alpha=0.5)
axes[0].plot(x_plot, pred_tc_mean[sort_idx], color='#1565C0', lw=1, label='Predicted Mean', alpha=0.8)
axes[0].set_xlabel('Sample Index (sorted by actual Tc)', fontsize=12)
axes[0].set_ylabel('Curie Temperature (K)', fontsize=12)
axes[0].set_title('Uncertainty Quantification — Curie Temp', fontsize=13, fontweight='bold')
axes[0].legend(fontsize=10)

# Hc with error bars
sort_idx2 = np.argsort(yHc_test.values)
x_plot2 = np.arange(len(sort_idx2))
axes[1].fill_between(x_plot2,
                     (pred_hc_mean[sort_idx2] - 2*pred_hc_std[sort_idx2]),
                     (pred_hc_mean[sort_idx2] + 2*pred_hc_std[sort_idx2]),
                     alpha=0.3, color='#2E7D32', label='95% CI (±2σ)')
axes[1].scatter(x_plot2, yHc_test.values[sort_idx2], s=8, c='black', zorder=3, label='Actual', alpha=0.5)
axes[1].plot(x_plot2, pred_hc_mean[sort_idx2], color='#2E7D32', lw=1, label='Predicted Mean', alpha=0.8)
axes[1].set_xlabel('Sample Index (sorted by actual log Hc)', fontsize=12)
axes[1].set_ylabel('log₁₀(Coercivity)', fontsize=12)
axes[1].set_title('Uncertainty Quantification — Coercivity', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('/content/fig8_uncertainty.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Figure 8 saved: fig8_uncertainty.png")


In [ ]:
# ============================================================
# STEP 15: Uncertainty Calibration — What % of actuals fall within CI?
# ============================================================
# Check calibration: how many actual values fall within predicted ±1σ, ±2σ?

def calibration_check(y_true, y_pred_mean, y_pred_std, name):
    within_1sigma = np.mean(np.abs(y_true - y_pred_mean) < 1 * y_pred_std) * 100
    within_2sigma = np.mean(np.abs(y_true - y_pred_mean) < 2 * y_pred_std) * 100
    within_3sigma = np.mean(np.abs(y_true - y_pred_mean) < 3 * y_pred_std) * 100
    print(f"  {name}:")
    print(f"    Within ±1σ: {within_1sigma:.1f}% (ideal: 68.3%)")
    print(f"    Within ±2σ: {within_2sigma:.1f}% (ideal: 95.4%)")
    print(f"    Within ±3σ: {within_3sigma:.1f}% (ideal: 99.7%)")
    return within_1sigma, within_2sigma, within_3sigma

print("UNCERTAINTY CALIBRATION CHECK")
print("=" * 50)
cal_tc = calibration_check(yTc_test.values, pred_tc_mean, pred_tc_std, "Curie Temperature")
print()
cal_hc = calibration_check(yHc_test.values, pred_hc_mean, pred_hc_std, "log(Coercivity)")


In [ ]:
# ============================================================
# STEP 16: Virtual Synthesis — 1,000 Theoretical Alloys
# ============================================================
np.random.seed(42)

# Common elements only: Fe, Co, Ni, Mn, Al, Si
common_elements = ['Fe', 'Co', 'Ni', 'Mn', 'Al', 'Si']

# VEC and density dictionaries (standard values)
vec_dict = {'Fe': 8, 'Co': 9, 'Ni': 10, 'Mn': 7, 'Al': 3, 'Si': 4,
            'Ag': 1, 'Au': 1, 'B': 3, 'Ba': 2, 'Be': 2, 'Bi': 5, 'C': 4,
            'Ca': 2, 'Cd': 2, 'Ce': 3, 'Cl': 7, 'Cr': 6, 'Cu': 1, 'Dy': 3,
            'Er': 3, 'Eu': 2, 'F': 7, 'Ga': 3, 'Gd': 3, 'Ge': 4, 'H': 1,
            'He': 0, 'Hf': 4, 'Ho': 3, 'I': 7, 'In': 3, 'Ir': 9, 'K': 1,
            'La': 3, 'Li': 1, 'Lu': 3, 'Mg': 2, 'Mo': 6, 'N': 5, 'Na': 1,
            'Nb': 5, 'Nd': 3, 'O': 6, 'Os': 8, 'P': 5, 'Pb': 4, 'Pd': 0,
            'Pr': 3, 'Pt': 1, 'Re': 7, 'Rh': 9, 'Ru': 8, 'S': 6, 'Sb': 5,
            'Sc': 3, 'Se': 6, 'Sm': 3, 'Sn': 4, 'Sr': 2, 'Ta': 5, 'Tb': 3,
            'Te': 6, 'Ti': 4, 'Tm': 3, 'U': 3, 'V': 5, 'W': 6, 'Y': 3,
            'Zn': 2, 'Zr': 4, 'Am': 3, 'As': 5, 'Pa': 3, 'Po': 6, 'Pu': 3,
            'Rb': 1}

rho_dict = {'Fe': 7.874, 'Co': 8.90, 'Ni': 8.908, 'Mn': 7.21, 'Al': 2.70, 'Si': 2.33,
            'Ag': 10.49, 'Au': 19.32, 'B': 2.34, 'Ba': 3.51, 'Be': 1.85, 'Bi': 9.78,
            'C': 2.27, 'Ca': 1.55, 'Cd': 8.65, 'Ce': 6.77, 'Cl': 3.21, 'Cr': 7.19,
            'Cu': 8.96, 'Dy': 8.55, 'Er': 9.07, 'Eu': 5.24, 'F': 1.70, 'Ga': 5.91,
            'Gd': 7.90, 'Ge': 5.32, 'H': 0.09, 'He': 0.18, 'Hf': 13.31, 'Ho': 8.80,
            'I': 4.93, 'In': 7.31, 'Ir': 22.56, 'K': 0.86, 'La': 6.15, 'Li': 0.53,
            'Lu': 9.84, 'Mg': 1.74, 'Mo': 10.22, 'N': 1.25, 'Na': 0.97, 'Nb': 8.57,
            'Nd': 7.01, 'O': 1.43, 'Os': 22.59, 'P': 1.82, 'Pb': 11.34, 'Pd': 12.02,
            'Pr': 6.77, 'Pt': 21.45, 'Re': 21.02, 'Rh': 12.41, 'Ru': 12.37, 'S': 2.07,
            'Sb': 6.68, 'Sc': 2.99, 'Se': 4.81, 'Sm': 7.52, 'Sn': 7.31, 'Sr': 2.64,
            'Ta': 16.65, 'Tb': 8.23, 'Te': 6.24, 'Ti': 4.51, 'Tm': 9.32, 'U': 19.10,
            'V': 6.11, 'W': 19.25, 'Y': 4.47, 'Zn': 7.13, 'Zr': 6.51, 'Am': 12.0,
            'As': 5.73, 'Pa': 15.37, 'Po': 9.20, 'Pu': 19.85, 'Rb': 1.53}

# Generate random compositions using Dirichlet distribution
# This ensures fractions sum to 1.0 and creates realistic alloy distributions
n_virtual = 1000

# Strategy: Fe-heavy with varying secondary elements
# Use Dirichlet with concentration params favoring Fe
alpha_base = np.array([5.0, 2.0, 1.5, 1.0, 0.5, 0.5])  # Fe-heavy
compositions = np.random.dirichlet(alpha_base, size=n_virtual)

# Round to 3 decimal places and re-normalize
compositions = np.round(compositions, 3)
compositions = compositions / compositions.sum(axis=1, keepdims=True)

# Build DataFrame matching training feature columns
virtual_data = {}

# Initialize all x_ columns to 0
all_x_cols = [c for c in X_train.columns if c.startswith('x_')]
for col in all_x_cols:
    virtual_data[col] = np.zeros(n_virtual)

# Fill in the common elements
for i, el in enumerate(common_elements):
    col_name = f'x_{el}'
    if col_name in all_x_cols:
        virtual_data[col_name] = compositions[:, i]

virtual_df = pd.DataFrame(virtual_data)

# Calculate VEC and Density
virtual_df['VEC'] = sum(
    virtual_df[f'x_{el}'] * vec_dict[el] 
    for el in common_elements if f'x_{el}' in virtual_df.columns
)
virtual_df['rho_theoretical (g/cm^3)'] = sum(
    virtual_df[f'x_{el}'] * rho_dict[el] 
    for el in common_elements if f'x_{el}' in virtual_df.columns
)

# Set categorical features: Soft Ferromagnetic
magnet_cols = [c for c in X_train.columns if c.startswith('Magnet_type_')]
for c in magnet_cols:
    virtual_df[c] = 1 if c == 'Magnet_type_Soft' else 0

magnetic_cols = [c for c in X_train.columns if c.startswith('Magnetic_type_')]
for c in magnetic_cols:
    virtual_df[c] = 1 if c == 'Magnetic_type_Ferromagnetic' else 0

# Ensure column order matches training data exactly
virtual_df = virtual_df[X_train.columns]

print(f"✅ Generated {n_virtual} virtual alloys")
print(f"   Columns: {virtual_df.shape[1]} (matches training: {virtual_df.shape[1] == X_train.shape[1]})")
print(f"   VEC range: [{virtual_df['VEC'].min():.2f}, {virtual_df['VEC'].max():.2f}]")
print(f"   Density range: [{virtual_df['rho_theoretical (g/cm^3)'].min():.2f}, {virtual_df['rho_theoretical (g/cm^3)'].max():.2f}]")


In [ ]:
# ============================================================
# STEP 17: Predict on Virtual Alloys WITH Uncertainty
# ============================================================

# --- Curie Temperature predictions (XGBoost primary, RF for UQ) ---
virtual_pred_tc_xgb = best_xgb_tc.predict(virtual_df)

# RF ensemble UQ for Curie Temp
virtual_tree_tc = np.array([tree.predict(virtual_df.values) for tree in rf_tc.estimators_])
virtual_tc_mean = virtual_tree_tc.mean(axis=0)
virtual_tc_std  = virtual_tree_tc.std(axis=0)

# --- Coercivity predictions (RF primary with UQ) ---
virtual_tree_hc = np.array([tree.predict(virtual_df.values) for tree in best_rf_hc.estimators_])
virtual_hc_mean = virtual_tree_hc.mean(axis=0)
virtual_hc_std  = virtual_tree_hc.std(axis=0)

# Convert from log space
virtual_hc_oe = 10 ** virtual_hc_mean
virtual_hc_oe_upper = 10 ** (virtual_hc_mean + 2*virtual_hc_std)
virtual_hc_oe_lower = 10 ** (virtual_hc_mean - 2*virtual_hc_std)

# Build results table
virtual_results = pd.DataFrame({
    'Fe': compositions[:, 0],
    'Co': compositions[:, 1],
    'Ni': compositions[:, 2],
    'Mn': compositions[:, 3],
    'Al': compositions[:, 4],
    'Si': compositions[:, 5],
    'VEC': virtual_df['VEC'].values,
    'Density': virtual_df['rho_theoretical (g/cm^3)'].values,
    'Pred_Tc_XGB': virtual_pred_tc_xgb,
    'Pred_Tc_RF_mean': virtual_tc_mean,
    'Pred_Tc_RF_std': virtual_tc_std,
    'Pred_Hc_Oe': virtual_hc_oe,
    'Pred_Hc_upper_95': virtual_hc_oe_upper,
    'Pred_Hc_lower_95': virtual_hc_oe_lower,
    'Pred_logHc_std': virtual_hc_std,
})

# --- CANDIDATE SELECTION ---
# Criteria: Tc > 1000K, Hc < 100 Oe, low uncertainty
candidates = virtual_results[
    (virtual_results['Pred_Tc_XGB'] > 1000) &
    (virtual_results['Pred_Hc_Oe'] < 100)
].copy()

# Rank by Tc (descending), then by uncertainty (ascending)
candidates['confidence_score'] = candidates['Pred_Tc_XGB'] / (candidates['Pred_Tc_RF_std'] + 1)
candidates = candidates.sort_values('confidence_score', ascending=False)

print(f"✅ Candidates meeting Tc > 1000K & Hc < 100 Oe: {len(candidates)} out of {n_virtual}")
print(f"\n{'='*90}")
print("TOP 10 CANDIDATES (ranked by confidence = Tc / uncertainty)")
print(f"{'='*90}")
top10 = candidates.head(10)
for i, (idx, row) in enumerate(top10.iterrows()):
    comp = f"Fe{row['Fe']:.2f}-Co{row['Co']:.2f}-Ni{row['Ni']:.2f}-Mn{row['Mn']:.2f}-Al{row['Al']:.2f}-Si{row['Si']:.2f}"
    print(f"  #{i+1}: {comp}")
    print(f"       Tc = {row['Pred_Tc_XGB']:.0f} K (±{row['Pred_Tc_RF_std']:.0f}K), Hc = {row['Pred_Hc_Oe']:.1f} Oe, VEC = {row['VEC']:.2f}")


In [ ]:
# ============================================================
# STEP 18: Virtual Synthesis Visualization
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: All virtual alloys colored by Tc
sc = axes[0].scatter(virtual_results['VEC'], virtual_results['Pred_Tc_XGB'],
                     c=np.log10(virtual_results['Pred_Hc_Oe'].clip(lower=0.01)),
                     cmap='coolwarm_r', s=20, alpha=0.6, edgecolors='none')
# Highlight candidates
if len(candidates) > 0:
    axes[0].scatter(candidates['VEC'], candidates['Pred_Tc_XGB'],
                    s=80, facecolors='none', edgecolors='lime', linewidths=2, 
                    label=f'Candidates ({len(candidates)})', zorder=5)
    # Mark top 5
    top5 = candidates.head(5)
    axes[0].scatter(top5['VEC'], top5['Pred_Tc_XGB'],
                    s=150, marker='*', c='gold', edgecolors='black', linewidths=0.5,
                    label='Top 5', zorder=6)

axes[0].axhline(y=1000, color='red', linestyle='--', lw=1.5, alpha=0.7, label='Tc = 1000 K')
axes[0].set_xlabel('Valence Electron Concentration (VEC)', fontsize=13)
axes[0].set_ylabel('Predicted Curie Temperature (K)', fontsize=13)
axes[0].set_title('Virtual Alloy Screening', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10, loc='lower right')
plt.colorbar(sc, ax=axes[0], label='log₁₀(Predicted Hc)')

# Right: Top 5 candidates with uncertainty bars
if len(candidates) >= 5:
    top5 = candidates.head(5)
    labels = [f"Fe{r['Fe']:.0%}Co{r['Co']:.0%}\nNi{r['Ni']:.0%}Mn{r['Mn']:.0%}" 
              for _, r in top5.iterrows()]
    y_pos = np.arange(5)
    
    axes[1].barh(y_pos, top5['Pred_Tc_XGB'].values, 
                 xerr=2*top5['Pred_Tc_RF_std'].values,
                 height=0.5, color='#1565C0', alpha=0.8, capsize=5, edgecolor='black')
    axes[1].axvline(x=1000, color='red', linestyle='--', lw=1.5, alpha=0.7)
    axes[1].set_yticks(y_pos)
    axes[1].set_yticklabels(labels, fontsize=10)
    axes[1].set_xlabel('Predicted Curie Temperature (K)', fontsize=13)
    axes[1].set_title('Top 5 Candidates with 95% CI', fontsize=14, fontweight='bold')
else:
    axes[1].text(0.5, 0.5, 'No candidates found\nRelax criteria', ha='center', va='center', fontsize=14, transform=axes[1].transAxes)

plt.tight_layout()
plt.savefig('/content/fig9_virtual_screening.png', dpi=200, bbox_inches='tight')
plt.show()
print("✅ Figure 9 saved: fig9_virtual_screening.png")


In [ ]:
# ============================================================
# STEP 19: SHAP Waterfall — Why the Top Candidate Has High Tc
# ============================================================
# This shows HOW the AI arrived at its prediction for the best candidate

if len(candidates) > 0:
    top_idx = candidates.index[0]
    top_alloy = virtual_df.iloc[virtual_df.index.get_indexer([top_idx])].iloc[0:1]
    
    # Get SHAP values for this single prediction
    shap_single = explainer_tc.shap_values(top_alloy)
    
    plt.figure(figsize=(12, 6))
    shap.waterfall_plot(
        shap.Explanation(
            values=shap_single[0], 
            base_values=explainer_tc.expected_value,
            data=top_alloy.values[0],
            feature_names=X_train.columns.tolist()
        ),
        max_display=15,
        show=False
    )
    plt.title('SHAP Waterfall — Top Candidate Curie Temp Prediction', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig('/content/fig10_shap_waterfall.png', dpi=200, bbox_inches='tight')
    plt.show()
    print("✅ Figure 10 saved: fig10_shap_waterfall.png")
else:
    print("No candidates to explain — relax selection criteria")


In [ ]:
# ============================================================
# STEP 20: Feature Importance — VEC vs Density (Assignment Question)
# ============================================================
from sklearn.inspection import permutation_importance

# Permutation importance for Curie Temp (XGBoost)
perm_tc = permutation_importance(best_xgb_tc, X_test, yTc_test, 
                                  n_repeats=15, random_state=42, n_jobs=-1, scoring='r2')

# Permutation importance for Coercivity (RF)
perm_hc = permutation_importance(best_rf_hc, X_test, yHc_test,
                                  n_repeats=15, random_state=42, n_jobs=-1, scoring='r2')

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Tc importance
imp_tc = pd.DataFrame({
    'feature': X_test.columns,
    'importance': perm_tc.importances_mean,
    'std': perm_tc.importances_std
}).sort_values('importance', ascending=False).head(15)

colors_tc = ['#FFD700' if 'VEC' in f else '#FF6B35' if 'rho' in f else '#1565C0' for f in imp_tc['feature']]
axes[0].barh(imp_tc['feature'][::-1], imp_tc['importance'][::-1],
             xerr=imp_tc['std'][::-1], color=colors_tc[::-1], capsize=3, edgecolor='black', alpha=0.85)
axes[0].set_xlabel('Permutation Importance (ΔR²)', fontsize=12)
axes[0].set_title('Feature Importance — Curie Temperature\n(Gold=VEC, Orange=Density)', fontsize=13, fontweight='bold')

# Hc importance
imp_hc = pd.DataFrame({
    'feature': X_test.columns,
    'importance': perm_hc.importances_mean,
    'std': perm_hc.importances_std
}).sort_values('importance', ascending=False).head(15)

colors_hc = ['#FFD700' if 'VEC' in f else '#FF6B35' if 'rho' in f else '#2E7D32' for f in imp_hc['feature']]
axes[1].barh(imp_hc['feature'][::-1], imp_hc['importance'][::-1],
             xerr=imp_hc['std'][::-1], color=colors_hc[::-1], capsize=3, edgecolor='black', alpha=0.85)
axes[1].set_xlabel('Permutation Importance (ΔR²)', fontsize=12)
axes[1].set_title('Feature Importance — log(Coercivity)\n(Gold=VEC, Orange=Density)', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('/content/fig11_feature_importance.png', dpi=200, bbox_inches='tight')
plt.show()

# Answer the assignment question
vec_rank_tc = list(imp_tc['feature']).index('VEC') + 1 if 'VEC' in list(imp_tc['feature']) else 'N/A'
rho_rank_tc = [i+1 for i, f in enumerate(imp_tc['feature']) if 'rho' in f]
vec_rank_hc = list(imp_hc['feature']).index('VEC') + 1 if 'VEC' in list(imp_hc['feature']) else 'N/A'
rho_rank_hc = [i+1 for i, f in enumerate(imp_hc['feature']) if 'rho' in f]

print(f"\n📊 ASSIGNMENT ANSWER: VEC vs Density")
print(f"   Curie Temp: VEC rank = #{vec_rank_tc}, Density rank = #{rho_rank_tc}")
print(f"   Coercivity: VEC rank = #{vec_rank_hc}, Density rank = #{rho_rank_hc}")
print("✅ Figure 11 saved: fig11_feature_importance.png")


In [ ]:
# ============================================================
# STEP 21: Final Summary — All Results
# ============================================================

print("=" * 70)
print("FINAL PIPELINE SUMMARY")
print("=" * 70)

print(f"\n📁 Dataset: {len(df)} samples, {X_train.shape[1]} features")
print(f"   Train: {len(X_train)}, Test: {len(X_test)}")

print(f"\n📊 Model Performance (Test Set):")
print(results.to_string(index=False, float_format='{:.4f}'.format))

if len(candidates) > 0:
    print(f"\n🧲 Virtual Synthesis: {n_virtual} alloys screened")
    print(f"   Candidates (Tc>1000K, Hc<100Oe): {len(candidates)}")
    print(f"\n🏆 TOP 5 PREDICTED SOFT MAGNET ALLOYS:")
    top5 = candidates.head(5)
    for i, (idx, row) in enumerate(top5.iterrows()):
        print(f"   #{i+1}: Fe={row['Fe']:.1%} Co={row['Co']:.1%} Ni={row['Ni']:.1%} "
              f"Mn={row['Mn']:.1%} Al={row['Al']:.1%} Si={row['Si']:.1%}")
        print(f"        Tc={row['Pred_Tc_XGB']:.0f}K (±{2*row['Pred_Tc_RF_std']:.0f}K), "
              f"Hc={row['Pred_Hc_Oe']:.1f} Oe, VEC={row['VEC']:.2f}")

print(f"\n📈 Figures Generated:")
for i in range(1, 12):
    print(f"   fig{i}_*.png")

print(f"\n✅ Pipeline complete. Ready for proposal.")


In [ ]:
# ============================================================
# STEP 22: Save All Outputs
# ============================================================
import joblib

# Save models
joblib.dump(best_xgb_tc, '/content/model_xgb_curie_temp.joblib')
joblib.dump(best_rf_hc, '/content/model_rf_coercivity_log.joblib')
joblib.dump(rf_tc, '/content/model_rf_curie_temp.joblib')
joblib.dump(xgb_hc, '/content/model_xgb_coercivity_log.joblib')

# Save virtual results
virtual_results.to_csv('/content/virtual_alloy_predictions.csv', index=False)
if len(candidates) > 0:
    candidates.to_csv('/content/top_candidates.csv', index=False)

# Save performance results
results.to_csv('/content/model_performance.csv', index=False)

print("✅ All models, predictions, and results saved to /content/")
print("\nSaved files:")
print("  - model_xgb_curie_temp.joblib")
print("  - model_rf_coercivity_log.joblib")
print("  - model_rf_curie_temp.joblib")
print("  - model_xgb_coercivity_log.joblib")
print("  - virtual_alloy_predictions.csv")
print("  - top_candidates.csv")
print("  - model_performance.csv")
print("  - fig1 through fig11 (PNG)")
